In [3]:
#upload raw step1 json file 
import json
from datasets import Dataset

with open('/scratch/hojinkim/mjgwak/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-1.full/train.5.6,21:33-5.json', 'r') as f:
    step1_full_raw = json.load(f)

dataset = Dataset.from_list(step1_full_raw)
dataset.push_to_hub('talzoomanzoo/0507-step1-full-raw', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]
No files have been modified since last commit. Skipping to prevent empty commit.
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-step1-full-metrics/commit/73dc01ed15d2ce42d07d81baf125e6b13010c1f6', commit_message='Upload dataset', commit_description='', oid='73dc01ed15d2ce42d07d81baf125e6b13010c1f6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-step1-full-metrics', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-step1-full-metrics'), pr_revision=None, pr_num=None)

In [5]:
#upload raw step1 json file 
import json
from datasets import Dataset

with open('/scratch/hojinkim/mjgwak/search-o1-dev/scripts/outputs/aime.ds-qwen-7b.direct.step-2.full/train.5.7,1:46-5.json', 'r') as f:
    step2_full_raw = json.load(f)

dataset = Dataset.from_list(step2_full_raw)
dataset.push_to_hub('talzoomanzoo/0507-step2-full-raw', private=False)

Creating parquet from Arrow format: 100%|██████████| 24/24 [00:00<00:00, 91.17ba/s]
/home/hojinkim/miniconda3/envs/reasoning/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.28s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-step2-full-raw/commit/8affe06605fd0f6a3ff509b36cbe3ace9974fef9', commit_message='Upload dataset', commit_description='', oid='8affe06605fd0f6a3ff509b36cbe3ace9974fef9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-step2-full-raw', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-step2-full-raw'), pr_revision=None, pr_num=None)

In [1]:
#full sft dataset generation
import json

with open('/scratch/hojinkim/mjgwak/search-o1-dev/data/0507_run1/raw_step2/train.5.7,1:46-5.json', 'r') as f:
    sft_data = json.load(f)

filtered_sft_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

count = 0
for data in sft_data:
    count += 1
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    if has_math_equal == True | False:
        entry = {
            'idx': count,
            'id': data.get('new_id', None),
            'input_x': '',
            'ground_truth': '',
            'output_z': '',
            'output_y': '',
            'metrics': []
        }

        # Collect all matching fields
        for key in data:
            if key.startswith('Question_'):
                entry['input_x'] = extract_x(data[key])
                entry['output_z'] = "<think>" + extract_z(data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = data[key]
            elif key.startswith('Output_'):
                entry['output_y'] = "</think>" + data[key]
            elif key.startswith('Metrics_'):
                entry['metrics'].append(data[key])
        
        filtered_sft_data.append(entry)

with open('train5.7,1:46-5.fullsft_data.json', 'w') as f:
    json.dump(filtered_sft_data, f, ensure_ascii=False, indent=4)

In [2]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(filtered_sft_data)
dataset.push_to_hub('talzoomanzoo/0507-fullsft-data', private=False)

/home/hojinkim/miniconda3/envs/reasoning/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 61.12ba/s]
/home/hojinkim/miniconda3/envs/reasoning/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.40s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-fullsft-data/commit/8cd1d2f6e5f30e486b2d34cdd4a3794e2927205f', commit_message='Upload dataset', commit_description='', oid='8cd1d2f6e5f30e486b2d34cdd4a3794e2927205f', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-fullsft-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-fullsft-data'), pr_revision=None, pr_num=None)

In [3]:
#good sft dataset generation
import json

with open('/scratch/hojinkim/mjgwak/search-o1-dev/data/0507_run1/raw_step2/train.5.7,1:46-5.json', 'r') as f:
    sft_data = json.load(f)

filtered_sft_data = []

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

count = 0
for data in sft_data:
    count += 1
    has_math_equal = any(
        key.startswith('Metrics_') and data[key].get('math_equal', False)
        for key in data
    )
    
    if has_math_equal == True:
        entry = {
            'idx': count,
            'id': data.get('new_id', None),
            'input_x': '',
            'ground_truth': '',
            'output_z': '',
            'output_y': '',
            'metrics': []
        }

        # Collect all matching fields
        for key in data:
            if key.startswith('Question_'):
                entry['input_x'] = extract_x(data[key])
                entry['output_z'] = "<think>" + extract_z(data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = data[key]
            elif key.startswith('Output_'):
                entry['output_y'] = "</think>" + data[key]
            elif key.startswith('Metrics_'):
                entry['metrics'].append(data[key])
        
        filtered_sft_data.append(entry)

with open('train5.7,1:46-5.goodsft_data.json', 'w') as f:
    json.dump(filtered_sft_data, f, ensure_ascii=False, indent=4)

In [4]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(filtered_sft_data)
dataset.push_to_hub('talzoomanzoo/0507-goodsft-data', private=False)

Creating parquet from Arrow format: 100%|██████████| 10/10 [00:00<00:00, 45.04ba/s]
/home/hojinkim/miniconda3/envs/reasoning/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(


Uploading the dataset shards: 100%|██████████| 1/1 [00:06<00:00,  6.15s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-goodsft-data/commit/07855786e7f35e9ee53f458ed7bddeb5648d6af0', commit_message='Upload dataset', commit_description='', oid='07855786e7f35e9ee53f458ed7bddeb5648d6af0', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-goodsft-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-goodsft-data'), pr_revision=None, pr_num=None)

In [5]:
#stage 1 dpo dataset generation
import json
import re

with open('/scratch/hojinkim/mjgwak/search-o1-dev/data/0507_run1/raw_step2/train.5.7,1:46-5.json', 'r') as f:
    stage1_dpo_data = json.load(f)

def clean_input_x_z(data):
    """
    Clean the input_x_z from the data, removing <｜User｜>, <｜Assistant｜>, <｜begin_of_sentence｜>, <｜think｜>
    """
    cleaned_data = re.sub(r'<｜begin▁of▁sentence｜><｜User｜>', '', data)
    cleaned_data = re.sub(r'<｜Assistant｜>', '', cleaned_data)
    cleaned_data = re.sub(r'</think>', '', cleaned_data)
    return cleaned_data

filtered_stage1_dpo_data = []

for i in range(0, len(stage1_dpo_data), 5):
    chunk = stage1_dpo_data[i:i+5]

    true_pairs = []
    false_pairs = []

    for data in chunk:
        for key in data:
            if key.startswith('Metrics_'):
                if data[key].get('math_equal', True):
                    true_pairs.append(data)
                else:
                    false_pairs.append(data)

    if true_pairs and false_pairs: #selected the first one among the pairs
        chosen_data = true_pairs[0]
        rejected_data = false_pairs[0]

        entry = {
        'idx': i+1, #chunk index as id
        'id': re.sub(r'(.*?_\d+)_\d+$', r'\1', chosen_data.get('new_id', None)),
        'input_x_z': '',
        'ground_truth': '',
        'chosen_y': '',
        'rejected_y': '',
    }
    
        for key in chosen_data:
            if key.startswith('Question_'):
                entry['input_x_z'] = clean_input_x_z(chosen_data[key])
            elif key.startswith('Answer_'):
                entry['ground_truth'] = chosen_data[key]
            elif key.startswith('Output_'):
                entry['chosen_y'] = "</think>" + chosen_data[key]

        for key in rejected_data:
            if key.startswith('Output_'):
                entry['rejected_y'] = "</think>" + rejected_data[key]

        filtered_stage1_dpo_data.append(entry)

with open('train5.7,1:46-5.stage1_dpo_data.json', 'w') as f:
    json.dump(filtered_stage1_dpo_data, f, ensure_ascii=False, indent=4)

In [55]:
from datasets import Dataset

dataset = Dataset.from_list(filtered_stage1_dpo_data)
dataset.push_to_hub('talzoomanzoo/0507-stage1-dpo-data', private=False)

Creating parquet from Arrow format: 100%|██████████| 4/4 [00:00<00:00, 10.62ba/s]
/home/hojinkim/miniconda3/envs/reasoning/lib/python3.10/site-packages/huggingface_hub/lfs.py:337: UserWarning: hf_transfer is enabled but does not support uploading from bytes or BinaryIO, falling back to regular upload
  warnings.warn(
Uploading the dataset shards: 100%|██████████| 1/1 [00:05<00:00,  5.33s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-stage1-dpo-data/commit/497e705e35de995d3cf1517a48fc859fb4c0bb7c', commit_message='Upload dataset', commit_description='', oid='497e705e35de995d3cf1517a48fc859fb4c0bb7c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-stage1-dpo-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-stage1-dpo-data'), pr_revision=None, pr_num=None)

In [6]:
#stage 2 dpo dataset generation
import json
import re

with open('/scratch/hojinkim/mjgwak/search-o1-dev/data/0507_run1/raw_step2/train.5.7,1:46-5.json', 'r') as f:
    stage2_dpo_data = json.load(f)

def extract_x(data):
    """
    Extract the input x from the data, between <｜User｜> and <｜Assistant｜>
    """
    return data.split("<｜User｜>")[1].split("<｜Assistant｜>")[0]

def extract_z(data):
    """
    Extract the output z from the data, between <think> and </think>
    """
    return data.split("<think>")[1].split("</think>")[0]

filtered_stage2_dpo_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(stage2_dpo_data), 25):
    chunk_25 = stage2_dpo_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.5 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.5 and false_pairs:
            subgroup_data.append((z_true_percentage, false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_data = None
        rejected_data = None
        
        for percentage, data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_data is None:
                chosen_data = data
            elif type_ == 'rejected' and rejected_data is None:
                rejected_data = data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_data and rejected_data:
            entry = {
                'idx': idx+1,
                'id': chosen_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'rejected_z': '',
            }
            
            # Process chosen data
            for key in chosen_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_data[key]
            
            # Process rejected data
            for key in rejected_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_data[key])
            
            filtered_stage2_dpo_data.append(entry)

with open('train5.7,1:46-5.stage2_dpo_data.json', 'w') as f:
    json.dump(filtered_stage2_dpo_data, f, ensure_ascii=False, indent=4)

In [53]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(filtered_stage2_dpo_data)
dataset.push_to_hub('talzoomanzoo/0507-stage2-dpo-data', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.13s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-stage2-dpo-data/commit/b65d4e33f57c95d64dcc6a4c3a23b6a6a6d12921', commit_message='Upload dataset', commit_description='', oid='b65d4e33f57c95d64dcc6a4c3a23b6a6a6d12921', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-stage2-dpo-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-stage2-dpo-data'), pr_revision=None, pr_num=None)

In [7]:
#3. x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
import json
import re

with open('/scratch/hojinkim/mjgwak/search-o1-dev/data/0507_run1/raw_step2/train.5.7,1:46-5.json', 'r') as f:
    z_y_combined_data = json.load(f)

filtered_z_y_combined_data = []

# Process data in chunks of 25 rows
for idx in range(0, len(z_y_combined_data), 25):
    chunk_25 = z_y_combined_data[idx:idx+25]
    subgroup_data = []  # Store (percentage, data) pairs for each subgroup
    subgroup_z_chosen_y_chosen = []
    subgroup_z_chosen_y_rejected = []
    subgroup_z_rejected_y_chosen = []
    subgroup_z_rejected_y_rejected = []
    
    # Process each subgroup of 5 rows within the 25-row chunk
    for i in range(0, len(chunk_25), 5):
        subgroup = chunk_25[i:i+5]
        true_pairs = []
        false_pairs = []
        
        # Count true and false pairs in this subgroup
        for data in subgroup:
            for key in data:
                if key.startswith('Metrics_'):
                    if data[key].get('math_equal', True):
                        true_pairs.append(data)
                    else:
                        false_pairs.append(data)
        
        # Calculate z_true_percentage for this subgroup
        z_true_percentage = len(true_pairs) / len(subgroup)
        
        # Store the percentage and corresponding data
        if 0.5 <= z_true_percentage < 1.0 and true_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'chosen'))
        elif 0 < z_true_percentage < 0.5 and false_pairs:
            subgroup_data.append((z_true_percentage, true_pairs[0], false_pairs[0], 'rejected'))
    
    # If we have data, find the highest and lowest percentage entries
    if subgroup_data:
        # Sort by percentage
        subgroup_data.sort(key=lambda x: x[0])
        
        # Get the highest percentage (chosen) and lowest percentage (rejected)
        chosen_z_data = None
        rejected_z_data = None
        chosen_z_chosen_y_data = None
        chosen_z_rejected_y_data = None
        rejected_z_chosen_y_data = None
        rejected_z_rejected_y_data = None
        
        for percentage, true_data, false_data, type_ in subgroup_data:
            if type_ == 'chosen' and chosen_z_data is None:
                chosen_z_data = true_data
                chosen_z_chosen_y_data = true_data
                chosen_z_rejected_y_data = false_data
            elif type_ == 'rejected' and rejected_z_data is None:
                rejected_z_data = false_data
                rejected_z_chosen_y_data = true_data
                rejected_z_rejected_y_data = false_data
        
        # If we have both chosen and rejected data, create an entry
        if chosen_z_data and rejected_z_data:
            #x, z_chosen, y_chosen under z _chosen, y_reject under z _chosen, z_reject, y_chosen under z _reject, y_reject under z _reject 
            entry = {
                'idx': idx+1,
                'id': chosen_z_data.get('new_id', None).split('_')[0],
                'input_x': '',
                'chosen_z': '',
                'chosen_y_under_chosen_z': '',
                'rejected_y_under_chosen_z': '',
                'rejected_z': '',
                'chosen_y_under_rejected_z': '',
                'rejected_y_under_rejected_z': ''
            }
            
            # Process chosen z data
            for key in chosen_z_data:
                if key.startswith('Question_'):
                    entry['input_x'] = extract_x(chosen_z_data[key])
                    entry['chosen_z'] = "<think>" + extract_z(chosen_z_data[key])
                if key.startswith('Answer_'):
                    entry['ground_truth'] = chosen_z_data[key]
            for key in chosen_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_chosen_z'] = "</think>" + chosen_z_chosen_y_data[key]
            for key in chosen_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_chosen_z'] = "</think>" + chosen_z_rejected_y_data[key]
            
            # Process rejected z data
            for key in rejected_z_data:
                if key.startswith('Question_'):
                    entry['rejected_z'] = "<think>" + extract_z(rejected_z_data[key])
            for key in rejected_z_chosen_y_data:
                if key.startswith('Output_'):
                    entry['chosen_y_under_rejected_z'] = "</think>" + rejected_z_chosen_y_data[key]
            for key in rejected_z_rejected_y_data:
                if key.startswith('Output_'):
                    entry['rejected_y_under_rejected_z'] = "</think>" + rejected_z_rejected_y_data[key]
            
            filtered_z_y_combined_data.append(entry)

with open('train5.7,1:46-5.z_y_combined_data.json', 'w') as f:
    json.dump(filtered_z_y_combined_data, f, ensure_ascii=False, indent=4)

In [66]:
#huggingface push
from datasets import Dataset

dataset = Dataset.from_list(filtered_z_y_combined_data)
dataset.push_to_hub('talzoomanzoo/0507-z-y-combined-data', private=False)

Uploading the dataset shards: 100%|██████████| 1/1 [00:03<00:00,  3.09s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/talzoomanzoo/0507-z-y-combined-data/commit/747372765a7a5fb0a0a4dab85a5ccbe316ae9781', commit_message='Upload dataset', commit_description='', oid='747372765a7a5fb0a0a4dab85a5ccbe316ae9781', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/talzoomanzoo/0507-z-y-combined-data', endpoint='https://huggingface.co', repo_type='dataset', repo_id='talzoomanzoo/0507-z-y-combined-data'), pr_revision=None, pr_num=None)